# PREPARING DATA FOR POSTGRESQL

After getting both the manabox collection and the scryfall API data, this is the preparation for loading the data into a PostgreSQL DB.

In [26]:
# IMPORTS
import pandas as pd
from pathlib import Path
import ast

# Loading the merged_collection.csv
def load_csv():
    data = Path('../data/processed/merged_collection.csv')
    df = pd.read_csv(data)
    return df

df = load_csv()

df.shape
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2005 entries, 0 to 2004
Data columns (total 89 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   name_x               2005 non-null   str    
 1   quantity             2005 non-null   int64  
 2   rarity_x             2005 non-null   str    
 3   set_code             2005 non-null   str    
 4   set_name_x           2005 non-null   str    
 5   foil_x               2005 non-null   str    
 6   scryfall_id          2005 non-null   str    
 7   condition            2005 non-null   str    
 8   language             2005 non-null   str    
 9   object               2005 non-null   str    
 10  id                   2005 non-null   str    
 11  oracle_id            2005 non-null   str    
 12  multiverse_ids       2005 non-null   str    
 13  mtgo_id              1846 non-null   float64
 14  arena_id             1371 non-null   float64
 15  tcgplayer_id         1993 non-null   float64
 16 

In [27]:
COLS = [
    'name_x', 'quantity', 'set_name_x', 'rarity','type_line', 'cmc', 'mana_cost','keywords', 
    'color_identity', 'colors','power', 'toughness', 'loyalty','legalities', 'edhrec_rank'
]

col = df[[c for c in COLS if c in df.columns]].copy()

col.rename(columns={
    'name_x':        'name',
    'set_name_x':    'set',
    'mana_cost':     'cost',
    'color_identity':'identity'
}, inplace=True)

col = col.fillna('')

print(col.columns.tolist())

['name', 'quantity', 'set', 'type_line', 'cmc', 'cost', 'keywords', 'identity', 'colors', 'power', 'toughness', 'loyalty', 'legalities', 'edhrec_rank']


In [28]:
CODE_TO_NAME = {'W': 'White', 'U': 'Blue', 'B': 'Black', 'R': 'Red', 'G': 'Green'}
COLORS = list(CODE_TO_NAME.keys())

def color_list(val) -> list[str]:
    if not val or val == '':
        return []
    if isinstance(val, list):
        return val
    if str(val).startswith('['):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return []
    return []

def color_parse(val) -> str:
    letters = color_list(val)
    if not letters:
        return 'Colorless'
    return ' | '.join(CODE_TO_NAME.get(c, c) for c in letters)

# Apply to colors and identity columns
col['colors'] = col['colors'].apply(color_parse)
col['identity'] = col['identity'].apply(color_parse)

In [29]:
col.info()
col

<class 'pandas.DataFrame'>
RangeIndex: 2005 entries, 0 to 2004
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   name         2005 non-null   str    
 1   quantity     2005 non-null   int64  
 2   set          2005 non-null   str    
 3   type_line    2005 non-null   str    
 4   cmc          2005 non-null   float64
 5   cost         2005 non-null   str    
 6   keywords     2005 non-null   str    
 7   identity     2005 non-null   str    
 8   colors       2005 non-null   str    
 9   power        2005 non-null   str    
 10  toughness    2005 non-null   str    
 11  loyalty      2005 non-null   object 
 12  legalities   2005 non-null   str    
 13  edhrec_rank  2005 non-null   object 
dtypes: float64(1), int64(1), object(2), str(10)
memory usage: 1.3+ MB


,name,quantity,set,type_line,cmc,cost,keywords,identity,colors,power,toughness,loyalty,legalities,edhrec_rank
0,Lunar Convocation,1,Bloomburrow,Enchantment,2.0,{W}{B},[],Black | White,Black | White,,,,"{'standard': 'legal', 'future': 'legal', 'hist...",3091.0
1,Corpseberry Cultivator,1,Bloomburrow,Creature — Squirrel Warlock,3.0,{1}{B/G}{B/G},['Forage'],Black | Green,Black | Green,2,3,,"{'standard': 'legal', 'future': 'legal', 'hist...",11434.0
2,Lunar Convocation,1,Bloomburrow,Enchantment,2.0,{W}{B},[],Black | White,Black | White,,,,"{'standard': 'legal', 'future': 'legal', 'hist...",3091.0
3,Stormcatch Mentor,1,Bloomburrow,Creature — Otter Wizard,2.0,{U}{R},"['Prowess', 'Haste']",Red | Blue,Red | Blue,1,1,,"{'standard': 'legal', 'future': 'legal', 'hist...",998.0
4,Hunter's Talent,1,Bloomburrow,Enchantment — Class,2.0,{1}{G},[],Green,Green,,,,"{'standard': 'legal', 'future': 'legal', 'hist...",3632.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2000,Goatnap,2,Lorwyn Eclipsed,Sorcery,3.0,{2}{R},[],Red,Red,,,,"{'standard': 'legal', 'future': 'legal', 'hist...",14667.0
2001,Warren Torchmaster,2,Lorwyn Eclipsed,Creature — Goblin Warrior,2.0,{1}{R},['Blight'],Red,Red,2,2,,"{'standard': 'legal', 'future': 'legal', 'hist...",15351.0
2002,Blanchwood Armor,2,Foundations,Enchantment — Aura,3.0,{2}{G},['Enchant'],Green,Green,,,,"{'standard': 'legal', 'future': 'legal', 'hist...",3137.0
2003,Efflorescence,2,Secrets of Strixhaven,Instant,3.0,{2}{G},['Infusion'],Green,Green,,,,"{'standard': 'legal', 'future': 'not_legal', '...",20334.0


In [30]:
def keywords_parse(val) -> str:
    if not val or val == '':
        return ''
    if isinstance(val, list):
        return ' | '.join(val)
    if str(val).startswith('['):
        try:
            lst = ast.literal_eval(val)
            return ' | '.join(lst)
        except (ValueError, SyntaxError):
            return ''
    return str(val)

col['keywords'] = col['keywords'].apply(keywords_parse)

In [31]:
SUPERTYPES = {'Legendary', 'Basic', 'Snow', 'Kindred', 'World', 'Elite'}
TYPES = ['Creature', 'Planeswalker', 'Land', 'Enchantment',
              'Artifact', 'Instant', 'Sorcery', 'Battle']

def parse_type_line(type_line):
    if not isinstance(type_line, str) or not type_line.strip():
        return pd.Series({
            'supertype': '', 'type': 'Other',
            'subtype': '', 'is_dfc': False, 'type_back': ''
        })

    # double faced cards
    is_dfc = '//' in type_line

    if is_dfc:
        front, back = [t.strip() for t in type_line.split('//', 1)]
    else:
        front, back = type_line.strip(), ''

    # type - subtype
    if '—' in front:
        type_part, subtype = [t.strip() for t in front.split('—', 1)]
    else:
        type_part, subtype = front.strip(), ''

    # supertype - type
    words = type_part.split()
    supertypes = [w for w in words if w in SUPERTYPES]
    types = [w for w in words if w in TYPES]

    return pd.Series({
        'supertype': ' | '.join(supertypes),
        'type': types[0] if types else 'Other',
        'subtype':   subtype,
        'is_dfc':    is_dfc,
        'type_back': back
    })

parsed = col['type_line'].apply(parse_type_line)
col = pd.concat([col, parsed], axis=1)
col.drop(columns=['type_line'], inplace=True)

In [32]:

FINAL_COLS = [
    'name', 'quantity', 'set', 'rarity', 'supertype','type', 'subtype','is_dfc', 'type_back', 'cmc', 'cost', 'colors', 'identity', 'keywords', 'power', 'toughness', 'loyalty', 'edhrec_rank'
]

collection = col[[c for c in FINAL_COLS if c in col.columns]]

collection

,name,quantity,set,supertype,type,subtype,is_dfc,type_back,cmc,cost,colors,identity,keywords,power,toughness,loyalty,edhrec_rank
0,Lunar Convocation,1,Bloomburrow,,Enchantment,,False,,2.0,{W}{B},Black | White,Black | White,,,,,3091.0
1,Corpseberry Cultivator,1,Bloomburrow,,Creature,Squirrel Warlock,False,,3.0,{1}{B/G}{B/G},Black | Green,Black | Green,Forage,2,3,,11434.0
2,Lunar Convocation,1,Bloomburrow,,Enchantment,,False,,2.0,{W}{B},Black | White,Black | White,,,,,3091.0
3,Stormcatch Mentor,1,Bloomburrow,,Creature,Otter Wizard,False,,2.0,{U}{R},Red | Blue,Red | Blue,Prowess | Haste,1,1,,998.0
4,Hunter's Talent,1,Bloomburrow,,Enchantment,Class,False,,2.0,{1}{G},Green,Green,,,,,3632.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2000,Goatnap,2,Lorwyn Eclipsed,,Sorcery,,False,,3.0,{2}{R},Red,Red,,,,,14667.0
2001,Warren Torchmaster,2,Lorwyn Eclipsed,,Creature,Goblin Warrior,False,,2.0,{1}{R},Red,Red,Blight,2,2,,15351.0
2002,Blanchwood Armor,2,Foundations,,Enchantment,Aura,False,,3.0,{2}{G},Green,Green,Enchant,,,,3137.0
2003,Efflorescence,2,Secrets of Strixhaven,,Instant,,False,,3.0,{2}{G},Green,Green,Infusion,,,,20334.0


In [33]:
collection.shape

(2005, 17)

In [34]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

table_name = "collection"
collection.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"{table_name} in database {os.getenv('DB_NAME')}")

collection in database mtg_dashboard
